In [ ]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

In [2]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
from ipywidgets import interact, IntSlider, fixed
import tqdm 

n_cooldown = 14
date = ''


initialise_or_create_database_at(f"./{date}_SNSPD{n_cooldown}.db")
import snspd
params = snspd.snspd(f'snspd{n_cooldown}.yaml')

# Set up experiment
exp_name = f'SNSPD{n_cooldown}_{date}'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Started new experiment


In [15]:
import importlib
importlib.reload(snspd)
params = snspd.snspd(f'snspd{n_cooldown}.yaml')

In [8]:
station = Station(config_file="friesland.yaml")
# dmm = station.load_instrument("dmm", revive_instance=True)
# yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
# MS = station.load_instrument("osc", revive_instance=True)
# pm100d = station.load_instrument("pm100d", revive_instance=True) 
# pm100usb = station.load_instrument("pm100usb", revive_instance=True) 
pms120 = station.load_instrument("pms120", revive_instance=True)
# tc = station.load_instrument("fridge", revive_instance=True)
# p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

In [16]:
pm100usb = station.load_instrument("pm100usb", revive_instance=True) 

2026-07-01 16:05:07,370 ¦ qcodes.instrument.instrument_base.com.visa ¦ ERROR ¦ visa ¦ _connect_and_handle_error ¦ 222 ¦ [pm100usb(Thorlabs_PM100USB)] Could not connect at USB0::0x1313::0x8078::P0033329::0::INSTR
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 218, in _connect_and_handle_error
    visa_handle, visabackend, resource_manager = self._open_resource(
                                                 ~~~~~~~~~~~~~~~~~~~^
        address, visalib
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 246, in _open_resource
    resource = resource_manager.open_resource(address)
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\highlevel.py", line 3302, in open_resource
    res.open(access_mode, open_timeout)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages

VisaIOError: VI_ERROR_NCIC (-1073807264): The interface associated with this session is not currently the controller in charge.

In [12]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

('TCPIP0::10.196.52.96::inst0::INSTR',
 'ASRL1::INSTR',
 'ASRL2::INSTR',
 'ASRL5::INSTR',
 'ASRL6::INSTR',
 'ASRL7::INSTR',
 'ASRL8::INSTR',
 'ASRL9::INSTR',
 'ASRL10::INSTR',
 'ASRL11::INSTR',
 'ASRL12::INSTR',
 'ASRL13::INSTR',
 'TCPIP0::10.196.50.27::inst0::INSTR',
 'USB0::0x05E6::0x2230::9010428::0::INSTR',
 'USB0::0x1313::0x8072::1906768::0::INSTR',
 'USB0::0x1313::0x8072::1913782::0::INSTR',
 'USB0::0x1313::0x8078::P0033329::0::INSTR')

In [21]:
my_instrument = rm.open_resource('USB0::0x1313::0x8072::1906768::0::INSTR')
print(my_instrument.query('*IDN?'))

VisaIOError: VI_ERROR_NCIC (-1073807264): The interface associated with this session is not currently the controller in charge.

The pm100usb doesn't seem to be working. I will try the measurement with only the pms120. 

In [22]:
pms120.power()

2026-07-01 16:10:39,875 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



1.23248922e-09

In [23]:
laser.power()

7.0

In [25]:
params.laser_get_standard(laser=laser)

Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm


In [26]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)


Laser enable status: True


PMS120 is currently connected to the 90% port of the beam splitter. 

In [27]:
pms120.power()

2026-07-01 16:12:09,431 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



0.00491006486

In [28]:
pms120.power()

0.00490274513

In [29]:
pms120.power()

0.00490379101

In [30]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


PMS120 is currently connected after the yellow attenautor, which is connected to the 10% port of the beam splitter. Laser -> splitter -> yellow attenuator -> pms120. Objective is to check the value of the yellow attenuator - it has not been touched since calibrated for cooldown 12. 

In [31]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

Laser enable status: True


In [32]:
pms120.power()

2026-07-01 16:16:05,289 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



4.94863741e-08

In [33]:
pms120.power()

5.26330943e-08

In [34]:
pms120.power()

4.94863741e-08

In [35]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


Calculate attenuation, using beam splitter values left in the snspd14.yaml file. 

In [37]:
power90 = (0.00491006486 + 0.00490274513 + 0.00490379101)/3
power10 = (4.94863741e-08 + 5.26330943e-08 + 4.94863741e-08)/3
attenuation = 10*np.log10((params.bs10/params.bs90*power90)/power10)
print(f'Attenuation: {attenuation}')

Attenuation: 39.54108244203008


So it was calibrated to 40dB. The last measurement in the calibration notebook before Cooldown 12 must have been the other attenuator. 